In [1]:
import numpy as np
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.output_parsers import JsonOutputParser
from llm_usage_tracker import setup_patch, print_usage_costs


In [2]:
pdf_loader = PyPDFLoader(file_path='perfect_Resume_i.pdf')
#
resume_data = [doc.page_content for doc in pdf_loader.load()]

# resume_text = "\n".join(resume_data).strip()
#
# resume_text = resume_text.lower()
"".join(resume_data).strip()

'Tina Huang \nhellotinah@gmail.com | 123-456-7890 | 1 BeepbeepBoop St. Philadelphia, PA 19104 \nEDUCATION AND HONORS  \nUniversity of Pennsylvania, School of Engineering and Applied Science Philadelphia, PA \nMasters in Computer Science (MCIT)  May 2020 \n• Cumulative GPA: 3.7/4.0 \n• Relevant Coursework: Machine Learning, Data Science Statistics, Big Data Analytics, Probability & Discrete Mathematics \n \nUniversity of Toronto, Trinity College Toronto, ON \nHonors Bachelor of Science with High Distinction (Pharmacology Specialist) May 2017 \n• Cumulative GPA: 3.8/4.0, Pharmacology GPA: 3.9/4.0, Dean’s List \n• St. Hildas’s Scholarship, Trinity Chancellor’s Scholarship ($1000 awarded for top 2% of program) 2015, 2016 \n \nPROFESSIONAL EXPERIENCE  \nGoldman Sachs Group, Inc. New York, NY \nData Science & Machine Learning Summer Analyst, Securities Division June 2019 - August 2019 \n• Utilized python to implement unsupervised machine learning techniques for time series anomaly detection 

In [3]:
jd_data = """
Objectives of this role
Design and develop machine learning algorithms and deep learning applications and systems for [Company X]

Solve complex problems with multilayered data sets, and optimize existing machine learning libraries and frameworks

Collaborate with data scientists, administrators, data analysts, data engineers, and data architects on production systems and applications

Identify differences in data distribution that could potentially affect model performance in real-world applications

Ensure algorithms generate accurate user recommendations

Stay up to date with developments in the machine learning industry

Responsibilities
Study and transform data science prototypes and apply appropriate machine learning algorithms and tools
Run machine learning tests and experiments, and document findings and results

Train, retrain, and monitor machine learning systems and models as needed

Construct optimized data pipelines to feed machine learning models

Consult with managers to determine and refine machine learning objectives

Extend existing machine learning libraries and frameworks
Skills and qualifications
Impeccable analytical and problem-solving skills

Extensive math and computer skills, with a deep understanding of probability, statistics, and algorithms

In-depth knowledge of machine learning frameworks, like Keras or PyTorch

Familiarity with data structures, data modeling, and software architecture

Excellent time management and organizational skills

Desire to learn

Preferred qualifications
Proven experience as a machine learning engineer or similar role
Familiarity with Python, Java, and R

Excellent communication and collaboration skills

Innovative mind with a passion for continuous learning

General knowledge of building machine learning systems

Bachelor’s degree (or equivalent) in computer science, mathematics, or related field
"""

### LLM

In [4]:
from langchain_core.prompts import ChatPromptTemplate

In [5]:
strict_prompt = ChatPromptTemplate.from_template("""
You are a strict senior technical recruiter at a top tech company.
Your job is to protect the hiring bar. You reject overqualified
candidates for wrong roles and underqualified ones equally.

STRICT RULES YOU MUST FOLLOW:
- If the resume is for a completely different domain than the JD, overall_score must be below 30
- If more than 50% of required keywords are missing, overall_score cannot exceed 50
- If experience is less than required or irrelevant, experience_match must be below 40
- "Strong Yes" only if overall_score >= 80 AND keyword match >= 70%
- "Yes" only if overall_score >= 65
- "Maybe" only if overall_score >= 45
- "No" if overall_score < 45
- Never give benefit of doubt. Score only what is explicitly present in the resume.

SCORING CONTEXT (pre-calculated, use this to anchor your scores):
- Keyword match rate: {kw_score}% ({matched_count} of {total_keywords} keywords found)
- Matched keywords: {matched_keywords}
- Missing critical keywords: {missing_keywords}

Score this resume against the job description.
Return ONLY this JSON, no explanation:
{{
  "overall_score": <0-100 int>,
  "skills_match": <0-100 int>,
  "experience_match": <0-100 int>,
  "education_match": <0-100 int>,
  "shortlist_verdict": "Strong Yes" | "Yes" | "Maybe" | "No",
  "domain_match": true | false,
  "keyword_match_pct": <use the pre-calculated value above>,
  "missing_skills": [<top 5 most critical missing skills>],
  "strong_points": [<what actually impressed you>],
  "improvement_tips": [<specific actionable resume fixes>],
  "recruiter_note": "<one honest sentence a recruiter would say>"
}}

RESUME:
<resume>
{resume}
</resume>

JOB DESCRIPTION:
<jd>
{jd}
</jd>
""")


In [6]:
key_word_prompt = ChatPromptTemplate.from_template(
    """Extract all technical skills, tools, frameworks,
            and domain keywords from this job description.
            Return ONLY a JSON array of strings. No explanation.

            JD: {jd}"""
)

In [7]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
load_dotenv()

True

In [8]:
GROQ_API_KEY = os.getenv('GROQ_KEY')
OPEN_ROUTER_KEY = os.getenv('OPEN_ROUTER_KEY')
groq = ChatGroq(api_key=GROQ_API_KEY,model='llama-3.3-70b-versatile')
json_parser = JsonOutputParser()

In [9]:
import os
from langchain_openai import ChatOpenAI

OPEN_ROUTER_KEY = os.getenv('OPEN_ROUTER_KEY')

gpt_120b = ChatOpenAI(
    api_key='sk-live-a6835be54727f211f75b7461ad31713949ed51b31087965441cf7fb69413437b',
    model='openai/gpt-oss-120b',
    temperature=0,
    base_url="https://api.aicredits.in/v1"
)

response = gpt_120b.invoke("hello world")
print(response)

content='Hello! 👋 How can I assist you today?' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 51, 'prompt_tokens': 56, 'total_tokens': 107, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': 0, 'reasoning_tokens': 33, 'rejected_prediction_tokens': None, 'image_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0, 'cache_write_tokens': 0, 'video_tokens': 0}, 'cost': 1.1874e-05, 'is_byok': False, 'cost_details': {'upstream_inference_cost': 1.1874e-05, 'upstream_inference_prompt_cost': 2.184e-06, 'upstream_inference_completions_cost': 9.69e-06}}, 'model_provider': 'openai', 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': None, 'id': 'gen-1778077656-JZr8BictGXfTpyULZo93', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019dfdaf-ee55-7f83-a2a2-be62d74bd954-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 56, 'output_tokens': 51, 'total_tokens': 107, 'in

In [10]:
chain = strict_prompt |groq | json_parser
key_word_chain = key_word_prompt | groq | json_parser

In [11]:
keyword = key_word_chain.invoke({'jd':jd_data})
keyword = [key.lower() for key in keyword]
keyword

['machine learning',
 'deep learning',
 'algorithms',
 'data science',
 'data structures',
 'data modeling',
 'software architecture',
 'keras',
 'pytorch',
 'python',
 'java',
 'r',
 'probability',
 'statistics']

In [12]:
matched_keyword = [kw for kw in keyword if kw in resume_text]
mis_matched_keyword = [kw for kw in keyword if kw not in resume_text]
match_score = (len(matched_keyword) / len(keyword)) * 100
print(match_score*100,len(matched_keyword),len(mis_matched_keyword))

NameError: name 'resume_text' is not defined

In [ ]:
# setup_patch()
# llm_response = groq.invoke(strict_prompt.invoke({
#     'resume': resume_data,'jd': jd_data,
#     'keyword':keyword,
#     'kw_score':match_score,
#     'matched_count':len(matched_keyword),
#     'total_keywords':len(keyword),
#     'matched_keywords':matched_keyword,
#     'missing_keywords':mis_matched_keyword,
# }))
# json_parser.invoke(llm_response)

In [ ]:
# input_cost_per_token = llm_response.response_metadata['token_usage']['pricing_table']['llama-3.3-70b-versatile']['input_cost_per_token']
# output_cost_per_token = llm_response.response_metadata['token_usage']['pricing_table']['llama-3.3-70b-versatile']['output_cost_per_token']
#
# llm_response.response_metadata['token_usage']['prompt_tokens'] * input_cost_per_token,llm_response.response_metadata['token_usage']['completion_tokens'] * output_cost_per_token

In [ ]:
response = chain.invoke({
    'resume': resume_data,'jd': jd_data,
    'keyword':keyword,
    'kw_score':match_score,
    'matched_count':len(matched_keyword),
    'total_keywords':len(keyword),
    'matched_keywords':matched_keyword,
    'missing_keywords':mis_matched_keyword,
})
response

{'overall_score': 58,
 'skills_match': 60,
 'experience_match': 50,
 'education_match': 90,
 'shortlist_verdict': 'Maybe',
 'domain_match': True,
 'keyword_match_pct': 35.0,
 'missing_skills': ['deep learning',
  'algorithms',
  'keras',
  'pytorch',
  'data structures'],
 'strong_points': ['strong educational background',
  'relevant experience in data science and machine learning',
  'proficiency in multiple programming languages'],
 'improvement_tips': ['highlight experience with deep learning frameworks',
  'emphasize knowledge of data structures and software architecture',
  'provide more details on machine learning projects'],
 'recruiter_note': 'Candidate has a strong foundation in machine learning and data science, but lacks experience with deep learning frameworks and key technologies like Keras and PyTorch.'}

In [ ]:
data = {'questions': [{'question': 'Can you explain how a variational autoencoder could be used to generate speech waveforms in a universal TTS system?', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Deep Generative Models (speech synthesis)', 'category': 'conceptual', 'question_domain': 'software', 'sub_domain': 'speech synthesis'}, {'question': 'What are common approaches to mitigate gender or cultural bias in a large language model used for voice assistants, and how would you evaluate their effectiveness while preserving user privacy?', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Bias mitigation in NLP', 'category': 'scenario', 'question_domain': 'software', 'sub_domain': 'NLP fairness'}, {'question': 'Describe the steps to containerize a PyTorch inference model and deploy it on a Kubernetes cluster using Helm charts.', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Kubernetes', 'category': 'practical', 'question_domain': 'software', 'sub_domain': 'container orchestration'}, {'question': 'How would you set up a simple CI/CD pipeline in GitHub Actions that runs unit tests, builds a Docker image, and pushes it to ECR whenever a new model version is committed?', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'CI/CD for ML workflows', 'category': 'practical', 'question_domain': 'software', 'sub_domain': 'DevOps / ML pipelines'}, {'question': 'Tell me about a time when you mentored a junior researcher or intern on an ML project. What challenges did you face and how did you ensure their contribution was integrated into the final product?', 'type': 'behavioral', 'difficulty': 'easy', 'skill_target': 'Mentoring / Coaching', 'category': 'behavioral', 'question_domain': 'software', 'sub_domain': 'team collaboration'}]}


In [ ]:
for i,question in enumerate(data['questions']):
    print(i,question)
    j = question['question_domain']+"/"+question['sub_domain']


0 {'question': 'Can you explain how a variational autoencoder could be used to generate speech waveforms in a universal TTS system?', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Deep Generative Models (speech synthesis)', 'category': 'conceptual', 'question_domain': 'software', 'sub_domain': 'speech synthesis'}
1 {'question': 'What are common approaches to mitigate gender or cultural bias in a large language model used for voice assistants, and how would you evaluate their effectiveness while preserving user privacy?', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Bias mitigation in NLP', 'category': 'scenario', 'question_domain': 'software', 'sub_domain': 'NLP fairness'}
2 {'question': 'Describe the steps to containerize a PyTorch inference model and deploy it on a Kubernetes cluster using Helm charts.', 'type': 'technical', 'difficulty': 'easy', 'skill_target': 'Kubernetes', 'category': 'practical', 'question_domain': 'software', 'sub_domain': 'container